# الدرس 12 - تقليل سجل المحادثة باستخدام مساحة عمل الوكيل

توضح هذه المفكرة كيفية إدارة السياق في المحادثات الطويلة باستخدام إطار عمل Microsoft Agent. مع تزايد المحادثات، يزداد عدد الرموز — مما يتجاوز في النهاية نافذة سياق النموذج. نتعامل مع هذا من خلال **نمط تلخيص السياق** و **مساحة عمل الوكيل** للذاكرة الدائمة.

## ما ستتعلمه:
1. **لماذا إدارة السياق مهمة**: فهم حدود الرموز ونوافذ السياق
2. **الوكلاء المدركون للسياق**: بناء وكلاء يديرون سياق محادثاتهم بأنفسهم
3. **نمط تلخيص السياق**: استخدام الأدوات لتكثيف سجل المحادثة
4. **مساحة عمل الوكيل**: ذاكرة دائمة تبقى بعد تقليل السياق

## المتطلبات السابقة:
- إعداد Azure OpenAI مع تكوين متغيرات البيئة
- فهم مفاهيم الوكيل الأساسية من الدروس السابقة


## الإعداد


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## لماذا إدارة السياق مهمة

لكل نموذج لغوي كبير نافذة **سياق** محدودة — الحد الأقصى لعدد الرموز التي يمكن معالجتها في طلب واحد. مع تقدم محادثة متعددة الأدوار:

- **عدد الرموز ينمو بشكل خطي** مع كل رسالة من المستخدم ورد المساعد.
- **رموز الموجه تهيمن على التكلفة** لأن التاريخ الكامل يُعاد إرساله في كل دور.
- في النهاية تتجاوز المحادثة **نافذة السياق** ويقوم النموذج إما بالقطع أو الخطأ.

### استراتيجيات إدارة السياق

| الاستراتيجية | كيف تعمل | المقايضة |
|---|---|---|
| **القطع** | حذف أقدم الرسائل | يفقد السياق المبكر |
| **الملخص** | تكثيف الرسائل الأقدم في ملخص | يفقد بعض التفاصيل، لكن تحتفظ بالنقاط الأساسية |
| **دفتر الملاحظات / الذاكرة الخارجية** | تخزين الحقائق الأساسية خارج المحادثة | يتطلب استدعاء أدوات، لكنه يبقى رغم أي تقليص |

في هذا الدفتر ندمج **الملخص** مع أداة **دفتر الملاحظات** حتى يتمكن الوكيل من الحفاظ على الاستمرارية حتى عندما يُكثّف تاريخ المحادثة.


## إنشاء وكيل مدرك للسياق


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## محاكاة محادثة طويلة

دعنا نتتبع محادثة متعددة الجولات لنرى كيف تتراكم السياقات. يجب أن يحتفظ الوكيل بالتفاصيل الرئيسية (التفضيلات، الميزانية، تواريخ السفر) عبر الجولات ويظهر الاستمرارية.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

لاحظ كيف يحتفظ الوكيل بالسياق من الأدوار السابقة — فهو يعرف عن اليابان، السوشي، المعابد، التصوير الفوتوغرافي، ميزانية 3000 دولار، السفر الفردي، وإطار الوقت في أبريل. في محادثة قصيرة، يعمل هذا جيدًا، ولكن مع نمو المحادثة يصبح إعادة إرسال التاريخ الكامل مكلفًا.

لنستمر في المحادثة مع المزيد من الأدوار لنرى تراكم السياق:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## نمط تلخيص السياق

مع نمو المحادثة، يمكننا استخدام **أداة التلخيص** لتكثيف السياق المتراكم في صيغة مختصرة. يستدعي الوكيل هذه الأداة لتسجيل التفضيلات الرئيسية بحيث حتى إذا تم حذف الرسائل الأقدم، يتم الحفاظ على المعلومات الأساسية.

يُعد هذا النمط لبنة البناء لتقليل السجل بشكل أكثر تطورًا:
1. يحدد الوكيل الحقائق الرئيسية من المحادثة
2. يستدعي أداة التلخيص لحفظها
3. يمكن إزالة الرسائل الأقدم بأمان لأن الملخص يلتقط ما هو مهم

فيما يلي نعرّف أداة `summarize_preferences` التي يمكن للوكيل استدعاؤها لتسجيل ملخص مضغوط لما تعلمه.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## الملخص

في هذا الدرس تعلمت كيفية إدارة السياق في المحادثات التي تستمر لفترة طويلة باستخدام إطار عمل Microsoft Agent:

### المفاهيم الرئيسية
- **نوافذ السياق محدودة** — كل توكن في تاريخ المحادثة يكلف مالًا ويُحتسب ضمن الحد الأقصى.
- **أدوات التلخيص** تتيح للوكيل تكثيف السياق المتراكم إلى ملخصات مضغوطة، مما يقلل من استخدام التوكن مع الحفاظ على المعلومات الأساسية.
- **ألواح ملاحظات الوكيل** توفر ذاكرة خارجية دائمة تبقى موجودة مهما تم تقليل المحادثة.

### ما الذي بنيته
- **وكيل مدرك للسياق** يحافظ على الاستمرارية عبر محادثات متعددة الأدوار
- **أداة تلخيص** (`summarize_preferences`) تسجل التفاصيل الرئيسية للمستخدم في صيغة مضغوطة
- **محادثة متعددة الأدوار** توضح الاحتفاظ بالسياق والتعامل مع التغييرات

### التطبيقات العملية
- **بوتات خدمة العملاء**: تتذكر التفضيلات عبر جلسات دعم طويلة
- **المساعدون الشخصيون**: يتابعون المشاريع الجارية دون الحاجة لإعادة شرح السياق
- **المدرسون التعليميون**: يحافظون على تقدم الطلاب عبر العديد من التفاعلات

### الخطوات التالية
- تنفيذ أداة لوحة ملاحظات كاملة مع الاحتفاظ القائم على الملفات
- إضافة قطع تلقائي للتاريخ بعد التلخيص
- الدمج مع قواعد بيانات المتجهات للبحث الدلالي في الذاكرة
- بناء وكلاء يمكنهم استئناف المحادثات بعد أيام مع الاحتفاظ بالسياق الكامل


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**إخلاء المسؤولية**:  
تمت ترجمة هذا المستند باستخدام خدمة الترجمة الآلية [Co-op Translator](https://github.com/Azure/co-op-translator). وعلى الرغم من جهودنا في تحقيق الدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الموثوق والمعتمد. للمعلومات الهامة والحساسة، يُنصح باستخدام الترجمة البشرية المهنية. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
